# Comparing 1D filtering methods

The [Filtering data in 1D](filtering_data_in_1D.ipynb) tutorial shows how to use `airbornegeo.filter_line` to low-pass filter survey lines. This notebook compiles a much wider range of ways to filter univariate data using several Python packages:

* [PyGMT](https://www.pygmt.org) (`pygmt.filter1d`, via `airbornegeo.filter_line`): gaussian, boxcar, cosine arch, and median filters
* [SciPy](https://scipy.org): IIR (Butterworth) and FIR frequency-domain filters, convolution kernels, Savitzky-Golay, Wiener, and smoothing splines
* [pandas](https://pandas.pydata.org): rolling windows (including time-based windows) and exponential moving averages
* [statsmodels](https://www.statsmodels.org): robust LOWESS and Kalman (state-space) smoothing
* direct FFT-domain filtering with `scipy.fft` (or the faster drop-in [pyFFTW](https://pyfftw.readthedocs.io))
* [ObsPy](https://docs.obspy.org): taper + zero-phase low-pass on seismology-style `Trace` objects
* [GSTools](https://geostat-framework.readthedocs.io): geostatistical filtering with kriging and a nugget effect
* [harmonica](https://www.fatiando.org/harmonica) (via `airbornegeo.eq_sources_1d`): physics-based equivalent sources

We test every method on the same data: the **raw free-air anomaly** from a line of the AGAP airborne gravity survey. Airborne gravimetry is a great stress test because the raw data is *extremely* noisy — aircraft vertical accelerations produce noise hundreds of times larger than the geological signal. We pay special attention to methods that cope with this extreme noise and methods that are **robust to outliers** (median-based filters, Hampel despiking, robust LOWESS).

In [ ]:
# %load_ext autoreload
# %autoreload 2

import gstools as gs
import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import statsmodels.api as sm
import verde as vd
from scipy import fft, ndimage, signal
from scipy.interpolate import make_smoothing_spline

import airbornegeo

## Load the data

We load the raw (unprocessed) AGAP gravity survey and keep the raw free-air anomaly (`Free_air`). The file also contains the free-air anomaly filtered by the original processors of the survey (`FAA_filt`); we keep it as a **reference** to compare our filters against.

We then extract a single line and calculate the distance along it.

In [ ]:
data_df = pd.read_csv("data/AGAP_gravity_survey.csv")
data_df = data_df[
    [
        "easting",
        "northing",
        "line",
        "unixtime",
        "Free_air",
        "FAA_filt",
        "Height_WGS1984",
    ]
]
data_df = data_df.rename(
    columns={"Free_air": "free_air", "FAA_filt": "faa_filt", "Height_WGS1984": "height"}
)
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)

# Create a Survey with the loaded data and add along-track distance
survey = airbornegeo.Survey(
    data_df,
    line_column="line",
    distance_column="distance_along_line",
)
survey.along_track_distance(progressbar=False)

# extract a single line from the survey
line_df = survey.data[survey.data.line == 4].copy()
# Create a Survey for this single line for filtering demonstrations
line_survey = airbornegeo.Survey(
    line_df,
    distance_column="distance_along_line",
)
line_survey.data.head()

## How noisy is this data?

Very. The standard deviation of the raw free-air anomaly is ~1300 mGal, while the filtered signal varies by less than ~100 mGal, i.e. the noise is more than an order of magnitude larger than the signal. In the top panel below the raw data (grey) completely swamps the reference signal (black); the bottom panel zooms the y-axis in on the signal.

Two properties of the sampling matter for the choice of filter:

* the line is sampled at 1 Hz (~63 m spacing at ~63 m/s ground speed), so treating samples as equally spaced is a good approximation ...
* ... except for one ~400 s gap in the middle of the line. Filters that work on the sample index simply smear across the gap, while methods that use the actual distance/time coordinate (PyGMT, time-based pandas windows, LOWESS, splines, equivalent sources) handle it correctly.

In [ ]:
distance_km = line_df.distance_along_line / 1e3

# find the gap in the line
time_steps = line_df.unixtime.diff()
gap_km = distance_km[time_steps > 10]

_fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
maxabs_raw = vd.maxabs(line_df.free_air, percentile=95)
ax0.plot(distance_km, line_df.free_air, ".", color="0.5", ms=1, label="raw free-air")
ax0.plot(
    distance_km, line_df.faa_filt, "k-", lw=0.8, label="survey-filtered (reference)"
)
ax0.set_ylim(-maxabs_raw, maxabs_raw)
ax0.set_ylabel("mGal")
ax0.legend(loc="lower left", markerscale=8)

maxabs = vd.maxabs(line_df.faa_filt) * 1.6
ax1.plot(distance_km, line_df.free_air, ".", color="0.5", ms=1)
ax1.plot(distance_km, line_df.faa_filt, "k-", lw=0.8)
ax1.set_ylim(-maxabs, maxabs)
ax1.set_ylabel("mGal")
ax1.set_xlabel("distance along line (km)")

noise_std = line_df.free_air.diff().std() / np.sqrt(2)
print(f"raw data standard deviation:      {line_df.free_air.std():.0f} mGal")
print(f"noise estimate from differences:  {noise_std:.0f} mGal")
print(f"reference signal std:             {line_df.faa_filt.std():.0f} mGal")

## Setting up a fair comparison

**Comparable filter widths:** every method parameterizes "smoothness" differently (a full filter width, a standard deviation, a -3 dB cutoff frequency, a window length, a penalty weight ...). To make the results roughly comparable we derive everything from a single choice: a cutoff wavelength of **19 km** (the same as in the other filtering tutorial). At the aircraft speed this corresponds to a cutoff period of ~300 s, or ~300 samples at 1 Hz. The equivalences are approximate — exactly matching the frequency responses of, say, a gaussian and a Butterworth filter is not possible.

**Padding:** filters misbehave near the edges of the data, and with noise this large the edge transients can be huge (we demonstrate this below with the Butterworth filter). `airbornegeo.filter_line` pads the line internally before calling PyGMT; for the other methods we do the same thing with a small helper that reflect-pads the data, filters, and trims the pad off again.

**Scoring:** for each method we record the RMS difference to the survey-supplied filtered free-air anomaly. **This measures agreement with the original processing, not accuracy** — the reference was itself produced by low-pass filtering this same raw data (it behaves like a ~19 km gaussian filter), so mean-based filters with a similar passband will agree closely, while a method can differ from the reference and still be a perfectly good filter. Treat the number as a sanity check plus a measure of "how similar to a gaussian low-pass is this?", and judge the plots as well.

In [ ]:
sampling_interval = line_df.unixtime.diff().median()  # seconds
speed = (
    line_df.distance_along_line.diff() / line_df.unixtime.diff()
).median()  # meters / second

cutoff_wavelength = 19_000  # meters
cutoff_period = cutoff_wavelength / speed  # seconds
window_samples = round(cutoff_period / sampling_interval)
if window_samples % 2 == 0:  # some filters require an odd window length
    window_samples += 1

print(f"sampling interval:  {sampling_interval:.0f} s")
print(f"ground speed:       {speed:.1f} m/s")
print(f"cutoff wavelength:  {cutoff_wavelength} m")
print(f"cutoff period:      {cutoff_period:.0f} s")
print(f"window length:      {window_samples} samples")

In [ ]:
results = {}


def filter_padded(values, filt, pad_width=500):
    """Reflect-pad the data, apply a filter function, and trim the pad off."""
    padded = np.pad(np.asarray(values, dtype=float), pad_width, mode="reflect")
    return filt(padded)[pad_width:-pad_width]


def score(name, values):
    """Store a filtered version of the line and its RMS difference to the reference."""
    values = np.asarray(values, dtype=float)
    error = airbornegeo.rmse(values - line_df.faa_filt)
    results[name] = {"values": values, "rmse": error}
    return error


def show_result(name, values):
    """Score a filtered version of the line and plot it against the reference."""
    error = score(name, values)
    _fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(distance_km, line_df.free_air, ".", color="0.5", ms=1, label="raw free-air")
    ax.plot(distance_km, line_df.faa_filt, "k-", lw=0.5, label="reference")
    ax.plot(
        distance_km, results[name]["values"], "-", color="tab:red", lw=1, label=name
    )
    ax.set_ylim(-maxabs, maxabs)
    ax.set_xlabel("distance along line (km)")
    ax.set_ylabel("mGal")
    ax.set_title(f"{name}: RMS difference to reference = {error:.1f} mGal")
    ax.legend(loc="lower left", markerscale=8)

## GMT filters with PyGMT

[`pygmt.filter1d`](https://www.pygmt.org/latest/api/generated/pygmt.filter1d.html) wraps GMT's `filter1d` module, which offers boxcar, cosine arch, gaussian, median, and mode filters (plus their high-pass complements). Its big advantage: it filters against an **arbitrary coordinate** (distance, time, ...), so it handles unevenly spaced data and gaps natively, and the filter width is specified directly in physical units (here: meters along the line).

`airbornegeo.filter_line` wraps it with automatic padding (and optional line-by-line grouping), so we use that with `engine="gmt"` — this is the same call as in the other filtering tutorial. Note for gaussian filters GMT defines the width as $6\sigma$.

In [ ]:
# Use the Survey API to filter the line
line_survey.filter_line(
    filter_width=cutoff_wavelength,  # 19 km low-pass gaussian filter
    filter_type="lowpass",
    filter_shape="gaussian",
    engine="gmt",
    data_column="free_air",
    result_column="gmt_gaussian_filtered",
    pad_width_percentage=10,
)
gmt_gaussian = line_survey.data["gmt_gaussian_filtered"]
show_result("GMT gaussian (pygmt)", gmt_gaussian)

The other GMT filter shapes use the same call — just swap `filter_shape`: `"boxcar"`, `"cosine"`, or `"median"` (airbornegeo doesn't expose GMT's mode filter). The **median filter is robust to outliers**: single wild values cannot pull it around, which makes it a good choice for spiky data. The price on this data is that it tracks the (mean-based) reference less closely, since the median and mean of very noisy data differ.

In [ ]:
gmt_filters = {
    "GMT boxcar (pygmt)": "boxcar",
    "GMT cosine arch (pygmt)": "cosine",
    "GMT median (pygmt)": "median",
}
_fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(distance_km, line_survey.data.faa_filt, "k-", lw=1.5, label="reference")
for (name, shape), color in zip(
    gmt_filters.items(), ["tab:blue", "tab:orange", "tab:green"], strict=True
):
    result_col = f"gmt_{shape}_filtered"
    line_survey.filter_line(
        filter_width=cutoff_wavelength,
        filter_type="lowpass",
        filter_shape=shape,
        engine="gmt",
        data_column="free_air",
        result_column=result_col,
        pad_width_percentage=10,
    )
    filtered = line_survey.data[result_col]
    error = score(name, filtered)
    ax.plot(
        distance_km,
        filtered,
        "-",
        color=color,
        lw=0.8,
        label=f"{name}: {error:.1f} mGal",
    )
ax.set_ylim(-maxabs, maxabs)
ax.set_xlabel("distance along line (km)")
ax.set_ylabel("mGal")
ax.legend(loc="lower left")

## Butterworth low-pass with SciPy

[`scipy.signal.butter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html) designs a classic infinite impulse response (IIR) low-pass filter with a maximally flat passband, specified by a **cutoff frequency** (the -3 dB / half-power point) and an **order** (roll-off steepness). Applying it forwards and backwards with [`sosfiltfilt`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfiltfilt.html) makes it zero-phase, so anomalies aren't shifted along the line. It assumes uniformly sampled data, so we work at 1 Hz and convert our cutoff period to a frequency. Chebyshev (`cheby1`/`cheby2`), elliptic (`ellip`), and Bessel (`bessel`) designs are drop-in replacements with different passband/roll-off trade-offs.

This is also a good place to demonstrate **why padding matters**: at such a low cutoff frequency the filter's impulse response is hundreds of samples long, much longer than `filtfilt`'s default edge handling. With noise this large, the result is enormous transients at the line ends.

In [ ]:
cutoff_freq = 1 / cutoff_period  # Hz
sampling_freq = 1 / sampling_interval  # Hz

sos = signal.butter(4, cutoff_freq, btype="lowpass", fs=sampling_freq, output="sos")

unpadded = signal.sosfiltfilt(sos, line_df.free_air)
padded = filter_padded(line_df.free_air, lambda x: signal.sosfiltfilt(sos, x))

_fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(distance_km, unpadded, "-", color="tab:orange", lw=1, label="without padding")
ax.plot(distance_km, padded, "-", color="tab:blue", lw=1, label="with reflect padding")
ax.set_ylim(-maxabs * 4, maxabs * 4)
ax.set_xlabel("distance along line (km)")
ax.set_ylabel("mGal")
ax.set_title("Edge transients from filtering without padding")
ax.legend(loc="lower left")

show_result("Butterworth low-pass (scipy)", padded)

## FIR low-pass with SciPy

[`scipy.signal.firwin`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html) designs a finite impulse response (FIR) filter with the window method. FIR filters are always stable and have exactly linear phase, but the number of taps controls how sharp the transition from passband to stopband is — and for a cutoff this low relative to the sampling rate (0.003 Hz at 1 Hz sampling!) you need a *lot* of taps. With too few taps the transition band is wider than the whole passband and the filter barely attenuates the near-cutoff noise.

In [ ]:
taps = signal.firwin(1801, cutoff_freq, fs=sampling_freq, window="blackman")
filtered = filter_padded(
    line_df.free_air,
    lambda x: signal.filtfilt(taps, 1.0, x),
    pad_width=1000,
)
show_result("FIR low-pass (scipy)", filtered)

## FFT-domain filtering with scipy.fft

Any linear filter can also be applied directly in the frequency domain: transform the (padded!) data with an FFT, multiply the spectrum by a transfer function, and transform back. This gives **total control over the frequency response**, costs only $O(n \log n)$, and stays cheap at very low cutoffs where an FIR filter needs thousands of taps. It is what GMT and harmonica do internally for their FFT-based filters, and it's exactly what `airbornegeo.filter_line`'s default `engine="scipy"` does too, so we call that directly rather than re-implementing it.

The one thing worth doing by hand is seeing what goes wrong without a smooth transfer function: don't use an ideal "brick-wall" cutoff — the sharp edge in the spectrum causes ringing (Gibbs oscillations) in space. `filter_line` doesn't offer a brick-wall shape (deliberately), so below we build one directly with `scipy.fft` to show why.

For very large datasets, [pyFFTW](https://pyfftw.readthedocs.io) is a faster drop-in replacement for `scipy.fft`.

In [ ]:
# Use the Survey API for scipy FFT filter
line_survey.filter_line(
    filter_width=cutoff_wavelength,
    filter_type="lowpass",
    filter_shape="gaussian",
    engine="scipy",
    data_column="free_air",
    result_column="fft_gaussian_scipy_filtered",
    pad_width_percentage=10,
)
show_result(
    "FFT-domain gaussian (airbornegeo, engine='scipy')",
    line_survey.data["fft_gaussian_scipy_filtered"],
)


def fft_brickwall(values):
    spacing = line_survey.data.distance_along_line.diff().median()  # meters
    freqs = fft.rfftfreq(len(values), d=spacing)
    transfer = (freqs <= 1 / cutoff_wavelength).astype(float)
    return fft.irfft(fft.rfft(values) * transfer, len(values))


error = score(
    "FFT-domain brick-wall (scipy.fft)",
    filter_padded(line_survey.data.free_air, fft_brickwall),
)
print(f"brick-wall cutoff: RMS difference = {error:.1f} mGal (ringing!)")

## Taper and low-pass with ObsPy

[ObsPy](https://docs.obspy.org) is the standard Python package for seismology. Its [`Trace`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.html) objects carry the sampling metadata and bundle the common time-series operations — [`filter`](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.filter.html) (low/high/band-pass, optionally `zerophase`), `taper`, `decimate`, `detrend` — as convenient methods. Under the hood the low-pass is the same SciPy Butterworth we used above, so the result is nearly identical; the value is the battle-tested, geophysics-oriented interface, especially if your data already lives in ObsPy streams.

In [ ]:
trace = obspy.Trace(data=np.pad(line_df.free_air.to_numpy(), 500, mode="reflect"))
trace.stats.sampling_rate = 1 / sampling_interval
trace.taper(max_percentage=0.05)
trace.filter("lowpass", freq=cutoff_freq, corners=4, zerophase=True)
show_result("Taper + lowpass (obspy)", trace.data[500:-500])

## Convolution kernels with scipy.ndimage

[`scipy.ndimage`](https://docs.scipy.org/doc/scipy/reference/ndimage.html) provides fast moving-window filters for uniformly sampled data: [`gaussian_filter1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter1d.html), [`uniform_filter1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.uniform_filter1d.html) (boxcar), and [`median_filter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.median_filter.html). These are the sample-index equivalents of the GMT filters above (using GMT's width $= 6\sigma$ convention for the gaussian) and they conveniently handle edges themselves (`mode="reflect"`). The **median filter** is again the robust option.

In [ ]:
sigma = window_samples / 6  # GMT's gaussian filter width convention
filtered = ndimage.gaussian_filter1d(
    line_df.free_air.to_numpy(), sigma=sigma, mode="reflect"
)
show_result("Gaussian kernel (scipy.ndimage)", filtered)

In [ ]:
filtered = ndimage.median_filter(
    line_df.free_air.to_numpy(), size=window_samples, mode="reflect"
)
show_result("Median filter (scipy.ndimage)", filtered)

## Savitzky-Golay with SciPy

[`scipy.signal.savgol_filter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.savgol_filter.html) fits a low-order polynomial to each window by least squares. Compared to a plain moving average it preserves peak shapes and widths much better (and can return smoothed derivatives for free), which makes it popular for spectroscopy-like data. It is a least-squares fit, so it is **not** robust to outliers.

In [ ]:
filtered = filter_padded(
    line_df.free_air,
    lambda x: signal.savgol_filter(x, window_length=window_samples, polyorder=2),
)
show_result("Savitzky-Golay (scipy)", filtered)

## Wiener filter with SciPy

[`scipy.signal.wiener`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.wiener.html) is an *adaptive* filter: in each window it compares the local variance to the noise level and smooths strongly where the data looks like pure noise but leaves high-variance regions (assumed to be signal) mostly alone. That logic breaks down here — the noise is so much larger than the signal that *every* window is high-variance, so the filter barely smooths at all. Included as an honest example of a method that is great on lightly noisy data (e.g. images) but the wrong tool when the noise dwarfs the signal.

In [ ]:
filtered = filter_padded(
    line_df.free_air, lambda x: signal.wiener(x, mysize=window_samples)
)
show_result("Wiener adaptive (scipy)", filtered)

## Rolling windows with pandas

pandas [`rolling`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html) windows can be specified as a **time offset** (e.g. `"301s"`) on a datetime index instead of a number of samples. The window then contains whatever samples fall inside the time span — so unevenly spaced data and our 400 s gap are handled correctly, unlike the sample-index methods above. `.mean()`, `.median()` (robust), `.std()`, or any custom function can be applied to the windows.

In [ ]:
free_air_ts = line_df.free_air.copy()
free_air_ts.index = pd.to_datetime(line_df.unixtime, unit="s")

rolling = free_air_ts.rolling(f"{cutoff_period:.0f}s", center=True)
show_result("Rolling mean, time-based (pandas)", rolling.mean())

In [ ]:
show_result("Rolling median, time-based (pandas)", rolling.median())

### Exponential moving average

[`ewm`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ewm.html) weights past samples with exponentially decaying weights. Unlike everything above it is **causal**: each value only uses data up to that point, which is what you want for real-time processing, but it lags behind the signal (the red curve is shifted to the right of the reference below) — the classic trade-off of one-sided filters.

In [ ]:
filtered = free_air_ts.ewm(
    halflife=f"{cutoff_period / 2:.0f}s", times=free_air_ts.index
).mean()
show_result("Exponential moving average (pandas)", filtered)

## LOWESS with statsmodels

[`statsmodels.api.nonparametric.lowess`](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html) (locally weighted scatterplot smoothing) fits a weighted linear regression in a window around every point. Two properties make it attractive here: it uses the actual **x coordinate** (so uneven spacing is fine), and the robustifying iterations (`it > 0`) **downweight outliers** using the residuals from the previous pass. The window is given as `frac`, the fraction of all data used around each point. It is $O(n^2)$-ish, so for long lines set `delta` (points closer than this reuse the last regression) to keep it fast.

In [ ]:
x = line_df.distance_along_line.to_numpy()
span = x.max() - x.min()

filtered = sm.nonparametric.lowess(
    line_df.free_air,
    x,
    frac=2 * cutoff_wavelength / span,  # window of ~2 cutoff wavelengths
    it=2,  # robustifying iterations to downweight outliers
    delta=0.002 * span,  # speed things up
    return_sorted=False,
)
show_result("LOWESS robust (statsmodels)", filtered)

## Smoothing splines with SciPy

[`scipy.interpolate.make_smoothing_spline`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.make_smoothing_spline.html) fits a cubic spline that balances misfit against curvature, controlled by the penalty weight `lam` — a continuous knob from pure interpolation (`lam=0`) to a straight line (`lam` huge). It also works directly on the x coordinate, and returns a *function*, so you can evaluate the smooth curve anywhere (handy for resampling).

A warning about the automatic choice: with `lam=None` the penalty is picked by generalized cross-validation, which assumes **uncorrelated** noise. Our noise is strongly correlated sample-to-sample, so GCV concludes the wiggles are signal and barely smooths at all — a very common failure mode of automatic smoothing selection on time series. Pick `lam` yourself by eye (its scale depends on the x units; here x is in meters, so useful values are huge).

In [ ]:
spline = make_smoothing_spline(x, line_df.free_air.to_numpy(), lam=1e12)
show_result("Smoothing spline (scipy)", spline(x))

gcv_spline = make_smoothing_spline(x, line_df.free_air.to_numpy())  # lam chosen by GCV
gcv_rmse = airbornegeo.rmse(gcv_spline(x) - line_df.faa_filt)
print(f"RMS difference with automatic (GCV) lam: {gcv_rmse:.0f} mGal (barely smooths!)")

## Kalman smoothing with statsmodels

State-space methods treat the signal as an unobserved state evolving in time and the data as noisy observations of it. [`statsmodels.api.tsa.UnobservedComponents`](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.structural.UnobservedComponents.html) implements such models — the `"smooth trend"` model (an integrated random walk) is a good default for a smoothly varying signal in heavy noise. The smoothness is controlled by two noise variances rather than a filter width.

The variances *can* be estimated from the data by maximum likelihood (`model.fit()`), but just like the GCV spline above, that estimation assumes uncorrelated observation noise — on this data it drives the trend variance to zero and returns a nearly straight line. So instead we set the variances ourselves with `model.smooth()`: the observation variance from our earlier first-difference noise estimate, and the trend variance from the target cutoff using the equivalence between this model and a smoothing spline, $\sigma^2_{trend} = \sigma^2_{obs} / (N_c / 2\pi)^4$ with $N_c$ the cutoff in samples.

Other selling points of state-space methods: missing values (NaNs) are handled natively, and besides the **smoothed state** used below (the best estimate using all data, i.e. two-sided and zero-phase) you also get the **filtered state** (`fit.filtered_state[0]`) — the causal, real-time estimate that only uses past data. With noise this heavy the causal estimate is dramatically noisier and laggier than the smoothed one, which is exactly why gravity surveys are filtered in post-processing rather than in real time.

In [ ]:
model = sm.tsa.UnobservedComponents(line_df.free_air.to_numpy(), "smooth trend")

observation_variance = noise_std**2
trend_variance = observation_variance / (window_samples / (2 * np.pi)) ** 4
fit = model.smooth([observation_variance, trend_variance])

show_result("Kalman smoother (statsmodels)", fit.smoothed_state[0])

## Kriging with GSTools

Geostatistics offers yet another view of filtering: treat the signal as a spatially **correlated random field** and the noise as an uncorrelated "nugget effect", then predict the field with kriging (equivalent to Gaussian-process regression). With `exact=False`, [GSTools](https://geostat-framework.readthedocs.io) does not honor the noisy observations exactly and the nugget variance is filtered out. Kriging's real strengths are that it predicts (with **uncertainty estimates**) at *any* location — sparse and irregular data are no problem — and that the smoothness comes from an interpretable covariance model (variance, correlation length, nugget).

Two practical caveats for using it as a *filter*:

* kriging solves a dense linear system, so it is $O(n^3)$ — on thousands of points, reduce the data first. We use 1 km **block medians** (also making the workflow robust to outliers).
* the nugget models the noise as *white* (uncorrelated). The noise here is strongly "blue" (its power grows with frequency, as vertical-acceleration noise does), so kriging is over-cautious and visibly under-predicts the anomaly amplitudes below. On data with genuinely white noise it does much better.

[PyKrige](https://geostat-framework.readthedocs.io/projects/pykrige) offers similar (2D/3D-focused) kriging, and [scikit-gstat](https://scikit-gstat.readthedocs.io) specializes in estimating the variogram (from which the covariance model parameters can be derived) rather than filtering itself.

In [ ]:
# reduce to 1 km block medians: robust to outliers and tames kriging's O(n^3) cost
block = (line_df.distance_along_line // 1000).astype(int)
blocked = line_df.groupby(block)[["distance_along_line", "free_air"]].median()

# covariance model: signal variance and correlation length, plus a nugget for the noise
block_noise_var = blocked.free_air.diff().var() / 2
signal_var = results["GMT gaussian (pygmt)"]["values"].var()
model = gs.Gaussian(
    dim=1, var=signal_var, len_scale=cutoff_wavelength, nugget=block_noise_var
)

krige = gs.krige.Ordinary(
    model,
    cond_pos=blocked.distance_along_line.to_numpy(),
    cond_val=blocked.free_air.to_numpy(),
    exact=False,  # don't honor the noisy observations exactly -> filters the nugget
)
filtered, kriging_variance = krige(line_df.distance_along_line.to_numpy())
show_result("Block median + kriging (gstools)", filtered)

## Equivalent sources with airbornegeo

If the data is a gravity or magnetic anomaly, we can exploit the physics: fit an equivalent-source model to the noisy data (with damping and block-averaging providing the smoothing) and predict at the same points — or at a higher elevation for extra (upward-continuation) smoothing. See the [Filtering data in 1D](filtering_data_in_1D.ipynb) tutorial for details. Unlike the generic filters, the result is guaranteed to be a physically plausible (harmonic) field.

In [ ]:
# Use the Survey API for eq_sources_1d
eqs = line_survey.eq_sources_1d(
    data_column="free_air",
    depth="default",
    damping=1e4,
    block_size=500,
)
filtered = eqs.predict(
    (
        line_survey.data.distance_along_line,
        np.zeros_like(line_survey.data.distance_along_line),
        line_survey.data.height,
    )
)
show_result("Equivalent sources (airbornegeo)", filtered)

## Dealing with outliers: despiking with a Hampel filter

The classic recipe for spiky data is to **despike first, then smooth**. The Hampel filter compares each sample to the median of a small window around it; samples more than a few (robust) standard deviations — estimated from the median absolute deviation (MAD) — away from the local median are declared outliers and replaced by that median. It's a few lines of pandas.

In [ ]:
def hampel(values, window=21, threshold=3):
    """Replace values more than threshold robust STDs from the local median."""
    values = pd.Series(np.asarray(values, dtype=float))
    rolling_median = values.rolling(window, center=True, min_periods=1).median()
    deviation = (values - rolling_median).abs()
    mad = deviation.rolling(window, center=True, min_periods=1).median()
    outliers = deviation > threshold * 1.4826 * mad  # 1.4826 converts MAD to STD
    return values.where(~outliers, rolling_median).to_numpy(), outliers.to_numpy()


despiked, outliers = hampel(line_df.free_air, window=51)
print(f"flagged {outliers.sum()} of {len(despiked)} samples as outliers")

filtered = ndimage.gaussian_filter1d(despiked, sigma=sigma, mode="reflect")
show_result("Hampel despike + gaussian", filtered)

Note the RMS difference to the reference got *worse* than the plain gaussian — because the reference kept those extreme samples, and here they are simply the tail of the (roughly symmetric) noise rather than genuine outliers, so removing them just moves us away from the reference. Despiking pays off when the spikes are *asymmetric or non-Gaussian* — instrument glitches, dropouts, steps. Let's build a controlled example of exactly that.

### A controlled outlier experiment

We take the smooth reference signal as ground truth, add moderate gaussian noise, and then corrupt 2% of the samples with large spikes. Since we now know the truth, the RMS errors are *real* accuracy numbers. To make the filters work harder we use a tighter smoothing (~1/5 of the width used above), so that a spike can't just be diluted away by a huge window.

In [ ]:
rng = np.random.default_rng(42)
n = len(line_df)
truth = line_df.faa_filt.to_numpy()

spiky = truth + rng.normal(0, 5, n)
spike_idx = rng.choice(n, size=int(0.02 * n), replace=False)
spiky[spike_idx] += rng.choice([-1, 1], spike_idx.size) * rng.uniform(
    100, 500, spike_idx.size
)

despiked, _ = hampel(spiky, window=21)
outlier_filters = {
    "gaussian kernel": ndimage.gaussian_filter1d(spiky, sigma=10, mode="reflect"),
    "median filter": ndimage.median_filter(spiky, size=61, mode="reflect"),
    "hampel + gaussian": ndimage.gaussian_filter1d(despiked, sigma=10, mode="reflect"),
    "LOWESS robust": sm.nonparametric.lowess(
        spiky, x, frac=120 / n, it=2, delta=0.002 * span, return_sorted=False
    ),
}

for name, filtered in outlier_filters.items():
    error = airbornegeo.rmse(filtered - truth)
    max_error = np.abs(filtered - truth).max()
    print(
        f"{name:20s} RMS error = {error:5.2f} mGal   max error = {max_error:5.1f} mGal"
    )

In [ ]:
_fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distance_km, spiky, ".", color="0.8", ms=2, label="spiky data")
ax.plot(distance_km, truth, "k-", lw=1.5, label="truth")
ax.plot(
    distance_km,
    outlier_filters["gaussian kernel"],
    "-",
    color="tab:orange",
    lw=1,
    label="gaussian kernel",
)
ax.plot(
    distance_km,
    outlier_filters["median filter"],
    "-",
    color="tab:blue",
    lw=1,
    label="median filter",
)
ax.set_xlim(60, 130)
ax.set_ylim(-120, 120)
ax.set_xlabel("distance along line (km)")
ax.set_ylabel("mGal")
ax.set_title("Mean-based filters smear spikes into bumps; robust filters ignore them")
ax.legend(loc="lower left", markerscale=4)

Every spike the gaussian filter touches becomes a bump in the output, while the median-based and robust methods are nearly unaffected. **If your data has outliers, either despike first or use a robust filter.**

## Comparing all methods

Back to the real raw free-air data. Remember the caveat from the setup section: these numbers measure **agreement with the survey's own (gaussian-like, mean-based) processing**, not accuracy. The median-based filters "score" worse mostly because they estimate a different statistic, and the causal exponential moving average scores worse because of its inherent lag — both can still be the right choice depending on your data and goal.

In [ ]:
summary = (
    pd.DataFrame(
        {
            "RMS difference to reference (mGal)": {
                k: v["rmse"] for k, v in results.items()
            }
        }
    )
    .sort_values("RMS difference to reference (mGal)")
    .round(1)
)
summary

In [ ]:
_fig, ax = plt.subplots(figsize=(8, 5.5))
plot_df = summary.iloc[::-1]
ax.barh(plot_df.index, plot_df.iloc[:, 0], color="tab:blue", height=0.65)
ax.set_xscale("log")
ax.set_xlabel("RMS difference to reference (mGal), log scale")
ax.spines[["top", "right"]].set_visible(False)
for i, value in enumerate(plot_df.iloc[:, 0]):
    ax.text(value * 1.1, i, f"{value:.1f}", va="center", fontsize=9)

In [ ]:
selected = [
    "Butterworth low-pass (scipy)",
    "LOWESS robust (statsmodels)",
    "Kalman smoother (statsmodels)",
    "Equivalent sources (airbornegeo)",
]
_fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distance_km, line_df.faa_filt, "k-", lw=1.8, label="reference")
for name, color in zip(
    selected, ["tab:blue", "tab:orange", "tab:green", "tab:purple"], strict=True
):
    ax.plot(distance_km, results[name]["values"], "-", color=color, lw=1, label=name)
ax.set_xlim(60, 130)
ax.set_ylim(-maxabs, maxabs)
ax.set_xlabel("distance along line (km)")
ax.set_ylabel("mGal")
ax.set_title("A closer look at a selection of methods")
ax.legend(loc="lower left")

## Summary

| Method | Function | Irregular sampling | Outlier-robust | Causal option | Notes |
|---|---|---|---|---|---|
| GMT gaussian / boxcar / cosine | `pygmt.filter1d` via `airbornegeo.filter_line` | yes | no | no | width in physical units; good default |
| GMT median | `pygmt.filter1d` via `airbornegeo.filter_line` | yes | **yes** | no | robust default for spiky data |
| Butterworth (IIR) | `scipy.signal.butter` + `sosfiltfilt` | no | no | yes (`sosfilt`) | precise frequency control; pad the ends! |
| FIR window design | `scipy.signal.firwin` + `filtfilt` | no | no | yes (`lfilter`) | needs many taps for low cutoffs |
| FFT-domain transfer function | `scipy.fft` (or pyFFTW) | no | no | no | total response control; pad + smooth transfer, no brick-walls |
| Taper + low-pass | `obspy` `Trace.filter` | no | no | yes (`zerophase=False`) | scipy under the hood; convenient for waveform workflows |
| Gaussian / boxcar kernel | `scipy.ndimage.gaussian_filter1d` | no | no | no | fastest; handles edges itself |
| Median filter | `scipy.ndimage.median_filter` | no | **yes** | no | fast robust workhorse |
| Savitzky-Golay | `scipy.signal.savgol_filter` | no | no | no | preserves peak shapes, gives derivatives |
| Wiener | `scipy.signal.wiener` | no | no | no | adaptive; needs noise ≪ signal |
| Rolling mean / median | `pandas` `.rolling("301s")` | **yes** (time windows) | median: **yes** | yes (`center=False`) | great for quick looks, gaps handled |
| Exponential moving average | `pandas` `.ewm` | yes (`times=`) | no | **always** | real-time use; lags |
| LOWESS | `statsmodels` `lowess` | yes | **yes** (`it>0`) | no | robust and flexible; slower |
| Smoothing spline | `scipy.interpolate.make_smoothing_spline` | yes | no | no | evaluate anywhere; don't trust GCV on correlated noise |
| Kalman smoother | `statsmodels` `UnobservedComponents` | no* | no | **yes** (filtered state) | set variances by hand or MLE; handles NaNs (*gaps as NaNs) |
| Hampel despike | pandas rolling median + MAD | no | **yes** | no | combine with any smoother |
| Block median + kriging | `gstools` (also `pykrige`) | **yes** | median step: **yes** | no | uncertainty estimates; nugget assumes white noise; $O(n^3)$ |
| Equivalent sources | `airbornegeo.eq_sources_1d` | **yes** | no | no | physics-based; gravity/magnetics only |

Some rules of thumb from this comparison:

* For along-line geophysical data, `airbornegeo.filter_line` (GMT gaussian) is a solid default: physical units, irregular sampling, automatic padding.
* **Always pad** (or use tools that do it for you) — with strong noise, edge transients can dominate everything else.
* If the data has spikes or outliers: despike with a Hampel filter first, or go straight to a median filter or robust LOWESS.
* For real-time / causal processing: exponential moving average (simple) or the Kalman filtered state (principled, handles missing data) — and accept the lag.
* For gravity and magnetic data, equivalent sources add physical consistency on top of smoothing, and can filter, upward-continue, and re-grid in one step.
* Kriging is the method of choice when the data is *sparse* and you need predictions with uncertainties at new locations; as a pure filter on densely sampled data with colored noise, the simpler mean-based filters track the signal better.

Other packages we assessed but did not demonstrate: [PyKrige](https://geostat-framework.readthedocs.io/projects/pykrige) (kriging with a 2D/3D focus — GSTools covers the 1D case shown here), [scikit-gstat](https://scikit-gstat.readthedocs.io) (variogram estimation and analysis rather than filtering), [GeostatsPy](https://github.com/GeostatsGuy/GeostatsPy) (GSLIB-style 2D geostatistical workflows), [NeuroDSP](https://neurodsp-tools.github.io) (neural time-series analysis; its filtering functions wrap the same `scipy.signal` FIR/IIR filters shown above), and [pyFFTW](https://pyfftw.readthedocs.io) (not a filter itself, but a faster FFT backend for the FFT-domain approach).